## Imports

In [ ]:
from pathlib import Path
import fitz
import re
from collections import Counter
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Done")

Done


### Detect project root

In [6]:
CURRENT_DIR = Path.cwd()

ROOT_DIR = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "data").is_dir() and (path / "config.py").is_file()
    ),
    None,
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root could not be detected.")

print("Project root:", ROOT_DIR)

Project root: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1


### Path setup

In [7]:
DATA_DIR = ROOT_DIR / "data"

KNOWLEDGE_BASE_DIR = DATA_DIR / "knowledge_base"
PLAYBOOK_DIR = DATA_DIR / "playbooks"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

KB_PDF_PATH = KNOWLEDGE_BASE_DIR / "PayPal_Dispute_Knowledge_Base.pdf"
VECTOR_STORE_DIR = ROOT_DIR / "rag" / "vector_store"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

assert KB_PDF_PATH.exists(), "Knowledge base PDF not found"

print("PDF:", KB_PDF_PATH)
print("Playbooks:", PLAYBOOK_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Vector store:", VECTOR_STORE_DIR)

PDF: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\knowledge_base\PayPal_Dispute_Knowledge_Base.pdf
Playbooks: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\playbooks
Processed data: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\processed
Vector store: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\rag\vector_store


### Creating required dictionaries and validate paths

In [8]:
PROCESSED_DATA_DIR.mkdir(parents = True, exist_ok = True)
VECTOR_STORE_DIR.mkdir(parents = True , exist_ok = True)

assert KB_PDF_PATH.exists(), "Knowledge base PDF not found"

print("PDF",KB_PDF_PATH)
print("playbboks:",PLAYBOOK_DIR)
print("proceesd Data :",PROCESSED_DATA_DIR)
print("vector store :",VECTOR_STORE_DIR)

PDF c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\knowledge_base\PayPal_Dispute_Knowledge_Base.pdf
playbboks: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\playbooks
proceesd Data : c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\processed
vector store : c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\rag\vector_store


### Loading the Knowledge Base

In [9]:
pdf_docs = []

with fitz.open(KB_PDF_PATH) as pdf:
    for page_number, page in enumerate(pdf, start=1):
        page_text = page.get_text("text").strip()
        if not page_text:
            continue
        pdf_docs.append({
            "text": page_text,
            "metadata": {
                "source": KB_PDF_PATH.name,   # was "sourse"
                "page": page_number,
                "document_type": "knowledge_base",
            },
        })

print("Total extracted pages:", len(pdf_docs))

Total extracted pages: 9


### Inspecting the extracted text

In [10]:
for idx, doc in enumerate(pdf_docs[:2]):
    print(f"idx: {idx} | metadata: {doc['metadata']}\n")
    print(doc["text"][:1000])
    print("=" * 80)

for doc in pdf_docs:
    print(f"Page: {doc['metadata']['page']} | Characters: {len(doc['text'])}")

pdf_text = "\n".join(doc["text"] for doc in pdf_docs)
policy_ids = sorted(set(re.findall(r"\b[A-Z]\d+\b", pdf_text)))
print("Policy references found:", policy_ids)

idx: 0 | metadata: {'source': 'PayPal_Dispute_Knowledge_Base.pdf', 'page': 1, 'document_type': 'knowledge_base'}

PayPal Dispute Knowledge Base
for PayPal Copilot (RAG corpus)  |  accessed 4 Nov 2026
Original summaries of public PayPal documentation. Not official PayPal text. Policies vary by country and change over time.
1
PayPal Dispute Knowledge Base
Item not received, not as described, unauthorized claims, evidence, Seller Protection and merchant
actions. Compiled for retrieval by the PayPal Copilot dispute agent.
How to use this document. Every entry is self-contained and starts with a metadata line: an ID, a type
(overview, api, policy, guidance or glossary), the dispute reason it applies to, and its source. Entries of type
guidance are the project builder's practical heuristics, not PayPal rules. Text is paraphrased from public
PayPal pages (accessed 4 Nov 2026); the United States Seller Protection policy is used throughout. Always
defer to PayPal's current published terms.
Cont

### Loading the playbooks

In [11]:
playbook_files = {
    "item_not_received": PLAYBOOK_DIR / "item_not_received.md",
    "not_as_described": PLAYBOOK_DIR / "not_as_described.md",
    "unauthorized": PLAYBOOK_DIR / "unauthorized.md",
}

def extract_playbooks(playbooks):
    docs = []
    for dispute_type, file_path in playbooks.items():
        if not file_path.exists():
            raise FileNotFoundError(f"Missing playbook: {file_path}")
        text = file_path.read_text(encoding="utf-8").strip()
        docs.append({
            "text": text,
            "metadata": {
                "source": file_path.name,
                "document_type": "playbook",
                "dispute_type": dispute_type,
            },
        })
    return docs

playbook_docs = extract_playbooks(playbook_files)
print("Playbooks:", len(playbook_docs))

Playbooks: 3


### Combine the documents

In [12]:
all_documents = pdf_docs + playbook_docs
print("Total documents:", len(all_documents))

PLACEHOLDERS = [
    "Write this in your own words",
    "(What the buyer is claiming",
    "(List the specific items",
    "(What looks suspicious",
    "(One short, factual example",
    "(Large amount, repeat buyer",
]

REQUIRED_SECTIONS = [
    "When this applies",
    "Evidence that wins",
    "Decision rule",
    "Evidence checklist",
    "Hand to a human when",
    "Sources",
]

def is_valid_playbook(text):
    if any(p in text for p in PLACEHOLDERS):
        return False
    return all(section in text for section in REQUIRED_SECTIONS)

documents_for_indexing = []
for document in all_documents:
    meta = document["metadata"]
    if meta["document_type"] == "playbook" and not is_valid_playbook(document["text"]):
        print("Skipping unfinished playbook:", meta["source"])
        continue
    documents_for_indexing.append(document)

print("Documents ready:", len(documents_for_indexing))
print("KB pages:", sum(d["metadata"]["document_type"] == "knowledge_base" for d in documents_for_indexing))
print("Playbooks:", sum(d["metadata"]["document_type"] == "playbook" for d in documents_for_indexing))

Total documents: 12
Skipping unfinished playbook: item_not_received.md
Skipping unfinished playbook: not_as_described.md
Skipping unfinished playbook: unauthorized.md
Documents ready: 9
KB pages: 9
Playbooks: 0


### chunking function

In [13]:
def fixed_size_chunk(text, chunk_size=900, overlap=150):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be between 0 and chunk_size")

    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# quick test
sample = (
    "PayPal disputes require merchants to provide relevant evidence.\n"
    "Tracking information can help establish whether an item was delivered.\n"
    "Merchants should respond before the seller response deadline."
)
for i, c in enumerate(fixed_size_chunk(sample, 100, 20), 1):
    print(f"Chunk {i}: {c}\n{'-' * 50}")

Chunk 1: PayPal disputes require merchants to provide relevant evidence.
Tracking information can help establ
--------------------------------------------------
Chunk 2: tion can help establish whether an item was delivered.
Merchants should respond before the seller re
--------------------------------------------------
Chunk 3: before the seller response deadline.
--------------------------------------------------


### build chunks 

In [14]:
chunks = []

for document in documents_for_indexing:
    split_texts = fixed_size_chunk(document["text"], chunk_size=900, overlap=150)

    for chunk_number, chunk_text in enumerate(split_texts):
        metadata = document["metadata"].copy()
        metadata["chunk_number"] = chunk_number

        # include the page so IDs are unique across PDF pages
        page_part = f"_p{metadata['page']}" if "page" in metadata else ""
        metadata["chunk_id"] = f"{metadata['source']}{page_part}_c{chunk_number}"

        chunks.append({"text": chunk_text, "metadata": metadata})

print("Total chunks:", len(chunks))

ids = [c["metadata"]["chunk_id"] for c in chunks]
assert len(ids) == len(set(ids)), "Duplicate chunk IDs found"
print(Counter(c["metadata"]["document_type"] for c in chunks))

Total chunks: 45
Counter({'knowledge_base': 45})


### embed and build the FAISS index 

In [15]:

EMBED_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in chunks]

embeddings = embedder.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True,   # unit vectors, so inner product = cosine similarity
).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # exact search, ideal for a corpus this small
index.add(embeddings)

print("Vectors in index:", index.ntotal)
print("Dimension:", dim)

# save to disk
faiss.write_index(index, str(VECTOR_STORE_DIR / "index.faiss"))

with open(VECTOR_STORE_DIR / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

with open(VECTOR_STORE_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({"embed_model": EMBED_MODEL, "dim": dim, "count": index.ntotal}, f, indent=2)

print("Saved to:", VECTOR_STORE_DIR)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Vectors in index: 45
Dimension: 384
Saved to: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\rag\vector_store


### search function and test

In [16]:
def search(query, k=4, document_type=None):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")

    # FAISS has no metadata filter on a flat index, so over-fetch and filter after
    fetch_k = index.ntotal if document_type else k
    scores, idxs = index.search(q_emb, fetch_k)

    results = []
    for score, i in zip(scores[0], idxs[0]):
        if i == -1:
            continue
        chunk = chunks[i]
        if document_type and chunk["metadata"]["document_type"] != document_type:
            continue
        results.append({"score": float(score), **chunk})
        if len(results) == k:
            break
    return results

def show(results):
    for r in results:
        print(f"[{r['metadata']['chunk_id']}] score={r['score']:.3f}")
        print(r["text"][:300].replace("\n", " "))
        print("-" * 60)

show(search("What happens when a buyer escalates an inquiry to a claim?"))

[PayPal_Dispute_Knowledge_Base.pdf_p2_c4] score=0.627
ad details and to respond. • reason: why the buyer opened the dispute (see PART B). • status and dispute_life_cycle_stage: whose turn it is and how far the case has progressed. • seller_response_due_date: the response deadline. • evidences: the evidence PayPal wants. Items whose source is REQUESTED_
------------------------------------------------------------
[PayPal_Dispute_Knowledge_Base.pdf_p7_c2] score=0.625
r requested, the buyer is asked to agree in the Resolution Center. If it equals or exceeds the request, the case resolves. E4. Escalate an inquiry to a claim ID: E4 | TYPE: api | REASON: MERCHANDISE_OR_SERVICE_NOT_RECEIVED, MERCHANDISE_OR_SERVICE_NOT_AS_DESCRIBED | SOURCE: Simulate test scenarios (s
------------------------------------------------------------
[PayPal_Dispute_Knowledge_Base.pdf_p3_c3] score=0.572
| SOURCE: Dispute reasons and evidence - developer.paypal.com/disputes/reasons-evidence ; Disputes Integration Gui

### reload from disk

In [17]:
import json
import faiss
from sentence_transformers import SentenceTransformer

with open(VECTOR_STORE_DIR / "config.json", encoding="utf-8") as f:
    cfg = json.load(f)

index = faiss.read_index(str(VECTOR_STORE_DIR / "index.faiss"))

with open(VECTOR_STORE_DIR / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

embedder = SentenceTransformer(cfg["embed_model"])

assert index.ntotal == len(chunks), "Index and chunks are out of sync"
print("Loaded", index.ntotal, "vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 45 vectors
